# Colab baseline v1

本 notebook 只做一件事：
- 重新建立 Colab 环境
- 跑一轮干净的 MAPPO baseline on VMAS/navigation
- 列出 outputs
- 用 metrics_summary.py 汇总结果


## 1. Clone repo

In [ ]:
!git clone https://github.com/WonderfulClaire/low_altitude_marl.git
%cd /content/low_altitude_marl
!ls

## 2. Install dependencies

In [ ]:
%cd /content/low_altitude_marl
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

## 3. Check torch and GPU

In [ ]:
import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))

## 4. Run MAPPO baseline (short run)

In [ ]:
%cd /content/low_altitude_marl
import os
os.environ['WANDB_MODE'] = 'disabled'

!python -m benchmarl.run \
  algorithm=mappo \
  task=vmas/navigation \
  experiment.render=false \
  experiment.evaluation=false \
  experiment.max_n_frames=60000 \
  seed=0

## 5. List outputs

In [ ]:
%cd /content/low_altitude_marl
!find outputs -maxdepth 3 -type d | sort

## 6. Summarize latest run
把下面 `RUN_ROOT` 改成上一个 cell 里最新生成的时间目录。

In [ ]:
%cd /content/low_altitude_marl
RUN_ROOT = '/content/low_altitude_marl/outputs/REPLACE_ME'
!python src/metrics_summary.py "$RUN_ROOT"

## 7. Optional: quick plot of episode_reward_mean

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path(RUN_ROOT)
target = None
for p in root.rglob('collection_reward_episode_reward_mean.csv'):
    target = p
    break
print('target:', target)
df = pd.read_csv(target, header=None, names=['step', 'value'])
plt.figure(figsize=(8,5))
plt.plot(df['step'], df['value'], marker='o')
plt.xlabel('step')
plt.ylabel('episode_reward_mean')
plt.title('MAPPO on VMAS Navigation: episode_reward_mean')
plt.grid(True)
plt.show()